- Pairs of R/S are kept intact across all splits and folds i.e., there's zero cross-fold leakage
- The splits will fully support a Tukey test. Each fold produces an independent test score because each test set contains entirely different molecules.
    - The pipeline produces 5 independent test scores per model (one per fold, on non-overlapping test sets), which is exactly the data structure Tukey HSD requires.
    - For each model, collect the test metric from each fold into a list per method, and then call `scipy.stats.tukey_hsd(scores_method_a, scores_method_b, ...)`.



In [1]:
import numpy as np
import pandas as pd

In [2]:
# load stereoisomer data
# https://github.com/jairesdesousa/chiraldlsv/blob/main/TSNE_maps/class_all.csv
input_path = "class_all.csv"
df = pd.read_csv(input_path, index_col=0)
df

,SMILES,SMILES_opp,TR/TE,F/L_class,@/@@_class,R/S_class
0,Brc1ccc2c(c1)N[C@H](c1ccccc1)CC2,Brc1ccc2c(c1)N[C@@H](c1ccccc1)CC2,TE,F,@,S
1,Brc1ccc2c(c1)N[C@@H](c1ccccc1)CC2,Brc1ccc2c(c1)N[C@H](c1ccccc1)CC2,TE,L,@@,R
2,C#CCO[C@H](CSc1nc2cc(Cl)ccc2s1)CN(C)C(c1ccccc1...,C#CCO[C@@H](CSc1nc2cc(Cl)ccc2s1)CN(C)C(c1ccccc...,TE,F,@,S
3,C#CCO[C@@H](CSc1nc2cc(Cl)ccc2s1)CN(C)C(c1ccccc...,C#CCO[C@H](CSc1nc2cc(Cl)ccc2s1)CN(C)C(c1ccccc1...,TE,L,@@,R
4,C=C(C(C)=O)[C@@H](CC(=O)c1ccc(Br)cc1)C(=O)OCC,C=C(C(C)=O)[C@H](CC(=O)c1ccc(Br)cc1)C(=O)OCC,TE,F,@@,R
...,...,...,...,...,...,...
3853,OC[C@]1(CCCOCc2ccccc2)COCCO1,OC[C@@]1(CCCOCc2ccccc2)COCCO1,TR,L,@,R
3854,OCCCC[C@H](O)c1ccc(F)cc1,OCCCC[C@@H](O)c1ccc(F)cc1,TR,F,@,S
3855,OCCCC[C@@H](O)c1ccc(F)cc1,OCCCC[C@H](O)c1ccc(F)cc1,TR,L,@@,R
3856,OCCCC[C@]1(CO)COCCO1,OCCCC[C@@]1(CO)COCCO1,TR,L,@,S


# Define helper functions

In [3]:
def make_kfold_splits(
    n_samples: int,
    n_folds: int,
    val_size: int,
    test_size: int,
    seed: int = 42,
):
    """
    Generate k-fold train/val/test index splits with zero overlap across val and test folds.

    Each molecule appears in exactly one validation set and one test set across
    all folds. This guarantees the independence required for statistical tests
    (e.g., Tukey HSD) when comparing models trained on different folds.

    Args:
        n_samples: Total number of molecules in the dataset.
        n_folds: Number of folds (typically 4 or 5).
        val_size: Number of samples in each validation split. Tip: estimate with int(n_samples * desired_fraction).
        test_size: Number of samples in each test split.
        seed: Random seed for reproducibility.

    Returns:
        A list of dicts of length n_folds. Each dict has keys "train", "val",
        and "test" mapping to 1D numpy arrays of integer indices into the
        original dataset.

    Example:
    >>> folds = make_kfold_splits(n_samples=3600, n_folds=5, val_size=252, test_size=252)
    """
    held_out_per_fold = val_size + test_size
    total_held_out = held_out_per_fold * n_folds

    if total_held_out > n_samples:
        raise ValueError(
            f"Cannot create {n_folds} folds with val_size={val_size} and "
            f"test_size={test_size}. Need {total_held_out} held-out slots but "
            f"only have {n_samples} samples. "
            f"Reduce n_folds, val_size, or test_size."
        )

    # shuffle once
    rng = np.random.default_rng(seed)
    indices = np.arange(n_samples)
    rng.shuffle(indices)

    # reserve the front of the shuffled array as the held-out pool.
    # each fold gets a non-overlapping slice of this pool.
    # remaining indices are always available for training.
    held_out_pool = indices[:total_held_out]
    always_train = indices[total_held_out:]

    folds = []
    for i in range(n_folds):
        start = i * held_out_per_fold
 
        val_indices  = held_out_pool[start : start + val_size]
        test_indices = held_out_pool[start + val_size : start + held_out_per_fold]
 
        # training = always_train + all held-out slices that don't belong to this fold
        other_held_out = np.concatenate([
            held_out_pool[:start],
            held_out_pool[start + held_out_per_fold:]
        ])
        train_indices = np.concatenate([always_train, other_held_out])
 
        assert len(val_indices) == val_size,  f"Fold {i}: val size mismatch"
        assert len(test_indices) == test_size, f"Fold {i}: test size mismatch"
        assert len(train_indices) == n_samples - held_out_per_fold, \
            f"Fold {i}: train size mismatch"
 
        folds.append({
            "train": train_indices,
            "val":   val_indices,
            "test":  test_indices,
        })
 
    return folds

In [4]:
def validate_splits(folds: list[dict[str, np.ndarray]], verbose: bool = True) -> bool:
    """
    Verify that val and test sets are non-overlapping across all folds.

    Checks three things:
      1. No overlap between val sets across folds.
      2. No overlap between test sets across folds.
      3. No molecule appears in both a val set and a test set across folds
         (within-fold splits are mutually exclusive by construction, but this check
         also catches cross-fold val-vs-test contamination).

    Args:
        folds: Output of make_kfold_splits(). A list of dicts, each with keys "train", "val", and "test".
        verbose: If True, prints a line-by-line summary of all pairwise overlap checks.

    Returns:
        True if all overlap checks pass (zero overlap everywhere),
        False otherwise.
    """
    n_folds = len(folds)
    all_clean = True

    for i in range(n_folds):
        val_i  = set(folds[i]["val"])
        test_i = set(folds[i]["test"])
        train_i = set(folds[i]["train"])

        # within-fold: no molecule should appear in more than one split
        within_overlaps = {
            "train ∩ val":  train_i & val_i,
            "train ∩ test": train_i & test_i,
            "val ∩ test":   val_i   & test_i,
        }
        for label, overlap in within_overlaps.items():
            if overlap:
                all_clean = False
                print(f"  ✗ Fold {i} within-fold {label}: {len(overlap)} overlapping indices!")
            elif verbose:
                print(f"  ✓ Fold {i} within-fold {label}: 0 overlap")
 
        # cross-fold: val and test sets must not repeat in other folds
        for j in range(i + 1, n_folds):
            val_j  = set(folds[j]["val"])
            test_j = set(folds[j]["test"])
 
            checks = {
                f"val[{i}]  ∩ val[{j}]":  (val_i,  val_j),
                f"test[{i}] ∩ test[{j}]": (test_i, test_j),
                f"val[{i}]  ∩ test[{j}]": (val_i,  test_j),
                f"test[{i}] ∩ val[{j}]":  (test_i, val_j),
            }
            for label, (a, b) in checks.items():
                overlap_count = len(a & b)
                pct = overlap_count / min(len(a), len(b)) * 100
                if overlap_count > 0:
                    all_clean = False
                    print(f"  ✗ {label}: {overlap_count} ({pct:.1f}%) overlapping indices!")
                elif verbose:
                    print(f"  ✓ {label}: 0 overlap")
 
    if verbose:
        status = "PASSED" if all_clean else "FAILED"
        print(f"\nOverall validation: {status}")
 
    return all_clean

In [5]:
def summarize_splits(folds: list[dict[str, np.ndarray]]) -> None:
    """
    Print a concise size summary of all folds.

    Args:
        folds: Output of make_kfold_splits(). A list of dicts, each with
            keys "train", "val", and "test".

    Returns:
        None
    """
    n = sum(len(f["train"]) + len(f["val"]) + len(f["test"]) for f in folds[:1])
    print(f'Total size: {n}')

    print(f"{'Fold':<6} {'Train':>8} {'Val':>8} {'Test':>8} {'Total':>8}")
    print("-" * 42)
    for i, fold in enumerate(folds):
        tr = len(fold["train"])
        v  = len(fold["val"])
        te = len(fold["test"])
        print(f"{i:<6} {tr:>8} {v:>8} {te:>8} {tr+v+te:>8}")

In [6]:
def expand_pair_indices(fold_indices: np.ndarray) -> np.ndarray:
    """
    Convert pair indices to molecule indices.

    Args:
        fold_indices: 1D array of pair indices (e.g., [0, 1, 5, ...]) where each value i represents the pair (2i, 2i+1).

    Returns:
        1D array of molecule indices with each pair index i expanded to molecule indices 2i and 2i+1.
    """
    return np.sort(np.concatenate([fold_indices * 2, fold_indices * 2 + 1]))

# Make Splits for 5 folds

In [7]:
df.shape

(3858, 6)

In [8]:
len(df) * 0.05

192.9

In [9]:
# must divide by two to keep the pairs together
folds = make_kfold_splits(n_samples=len(df) // 2, n_folds=5, val_size=190 // 2, test_size=190 // 2)
len(folds), len(folds[0]['train'])

(5, 1739)

In [10]:
validate_splits(folds, verbose=True)

  ✓ Fold 0 within-fold train ∩ val: 0 overlap
  ✓ Fold 0 within-fold train ∩ test: 0 overlap
  ✓ Fold 0 within-fold val ∩ test: 0 overlap
  ✓ val[0]  ∩ val[1]: 0 overlap
  ✓ test[0] ∩ test[1]: 0 overlap
  ✓ val[0]  ∩ test[1]: 0 overlap
  ✓ test[0] ∩ val[1]: 0 overlap
  ✓ val[0]  ∩ val[2]: 0 overlap
  ✓ test[0] ∩ test[2]: 0 overlap
  ✓ val[0]  ∩ test[2]: 0 overlap
  ✓ test[0] ∩ val[2]: 0 overlap
  ✓ val[0]  ∩ val[3]: 0 overlap
  ✓ test[0] ∩ test[3]: 0 overlap
  ✓ val[0]  ∩ test[3]: 0 overlap
  ✓ test[0] ∩ val[3]: 0 overlap
  ✓ val[0]  ∩ val[4]: 0 overlap
  ✓ test[0] ∩ test[4]: 0 overlap
  ✓ val[0]  ∩ test[4]: 0 overlap
  ✓ test[0] ∩ val[4]: 0 overlap
  ✓ Fold 1 within-fold train ∩ val: 0 overlap
  ✓ Fold 1 within-fold train ∩ test: 0 overlap
  ✓ Fold 1 within-fold val ∩ test: 0 overlap
  ✓ val[1]  ∩ val[2]: 0 overlap
  ✓ test[1] ∩ test[2]: 0 overlap
  ✓ val[1]  ∩ test[2]: 0 overlap
  ✓ test[1] ∩ val[2]: 0 overlap
  ✓ val[1]  ∩ val[3]: 0 overlap
  ✓ test[1] ∩ test[3]: 0 overlap
  ✓ val[1

True

In [11]:
# expand each split back to molecule indices
mol_folds = [
    {
        "train": expand_pair_indices(fold["train"]),
        "val":   expand_pair_indices(fold["val"]),
        "test":  expand_pair_indices(fold["test"]),
    }
    for fold in folds
]

In [12]:
validate_splits(mol_folds, verbose=True)

  ✓ Fold 0 within-fold train ∩ val: 0 overlap
  ✓ Fold 0 within-fold train ∩ test: 0 overlap
  ✓ Fold 0 within-fold val ∩ test: 0 overlap
  ✓ val[0]  ∩ val[1]: 0 overlap
  ✓ test[0] ∩ test[1]: 0 overlap
  ✓ val[0]  ∩ test[1]: 0 overlap
  ✓ test[0] ∩ val[1]: 0 overlap
  ✓ val[0]  ∩ val[2]: 0 overlap
  ✓ test[0] ∩ test[2]: 0 overlap
  ✓ val[0]  ∩ test[2]: 0 overlap
  ✓ test[0] ∩ val[2]: 0 overlap
  ✓ val[0]  ∩ val[3]: 0 overlap
  ✓ test[0] ∩ test[3]: 0 overlap
  ✓ val[0]  ∩ test[3]: 0 overlap
  ✓ test[0] ∩ val[3]: 0 overlap
  ✓ val[0]  ∩ val[4]: 0 overlap
  ✓ test[0] ∩ test[4]: 0 overlap
  ✓ val[0]  ∩ test[4]: 0 overlap
  ✓ test[0] ∩ val[4]: 0 overlap
  ✓ Fold 1 within-fold train ∩ val: 0 overlap
  ✓ Fold 1 within-fold train ∩ test: 0 overlap
  ✓ Fold 1 within-fold val ∩ test: 0 overlap
  ✓ val[1]  ∩ val[2]: 0 overlap
  ✓ test[1] ∩ test[2]: 0 overlap
  ✓ val[1]  ∩ test[2]: 0 overlap
  ✓ test[1] ∩ val[2]: 0 overlap
  ✓ val[1]  ∩ val[3]: 0 overlap
  ✓ test[1] ∩ test[3]: 0 overlap
  ✓ val[1

True

In [13]:
summarize_splits(mol_folds)

Total size: 3858
Fold      Train      Val     Test    Total
------------------------------------------
0          3478      190      190     3858
1          3478      190      190     3858
2          3478      190      190     3858
3          3478      190      190     3858
4          3478      190      190     3858


In [14]:
def validate_pairs(mol_folds: list[dict[str, np.ndarray]]) -> bool:
    """
    Check that every molecule index has its pair partner in the same split.

    Assumes that molecular pairs are always consecutive (i.e., index 0 and 1 for a pair, 2 and 3, etc.)
    This means the partner of any even index i is i+1, and the partner of any odd index i is i-1.

    Each even index i and its partner i+1 must always appear together in
    the same split (train, val, or test). If they were separated, a model
    could see one enantiomer during training and be evaluated on the other,
    causing data leakage.

    Args:
        mol_folds: Molecule-level folds from expand_pair_indices(). A list
            of dicts with keys "train", "val", and "test".

    Returns:
        True if all pairs are intact across all folds, False otherwise.
    """
    all_clean = True
    for fold_idx, mf in enumerate(mol_folds):
        for split_name in ["train", "val", "test"]:
            idx_set = set(mf[split_name])
            orphans = [
                idx for idx in idx_set
                if (idx + 1) not in idx_set  # even idx: partner is idx+1
                and (idx - 1) not in idx_set  # odd idx: partner is idx-1
            ]
            if orphans:
                print(f"  ✗ Fold {fold_idx} {split_name}: {len(orphans)} molecules missing their pair partner")
                all_clean = False
            else:
                print(f"  ✓ Fold {fold_idx} {split_name}: all pairs intact")
    return all_clean

In [15]:
validate_pairs(mol_folds)

  ✓ Fold 0 train: all pairs intact
  ✓ Fold 0 val: all pairs intact
  ✓ Fold 0 test: all pairs intact
  ✓ Fold 1 train: all pairs intact
  ✓ Fold 1 val: all pairs intact
  ✓ Fold 1 test: all pairs intact
  ✓ Fold 2 train: all pairs intact
  ✓ Fold 2 val: all pairs intact
  ✓ Fold 2 test: all pairs intact
  ✓ Fold 3 train: all pairs intact
  ✓ Fold 3 val: all pairs intact
  ✓ Fold 3 test: all pairs intact
  ✓ Fold 4 train: all pairs intact
  ✓ Fold 4 val: all pairs intact
  ✓ Fold 4 test: all pairs intact


True

In [16]:
def save_folds(folds: list[dict[str, np.ndarray]], path: str) -> None:
    """
    Save folds to a .npz file.

    Args:
        folds: Output of make_kfold_splits().
        path: Filepath to save to. Should end in .npz.

    Returns:
        None
    """
    arrays = {}
    for i, fold in enumerate(folds):
        arrays[f"fold{i}_train"] = fold["train"]
        arrays[f"fold{i}_val"]   = fold["val"]
        arrays[f"fold{i}_test"]  = fold["test"]
    np.savez(path, **arrays)

In [17]:
def load_folds(path: str) -> list[dict[str, np.ndarray]]:
    """Load folds from a .npz file.

    Args:
        path: Filepath to a .npz file saved by save_folds().

    Returns:
        A list of dicts with keys "train", "val", and "test", matching
        the output format of make_kfold_splits().

    """
    archive = np.load(path)
    
    # Infer n_folds from the keys
    fold_indices = sorted(set(
        int(k.split("_")[0].replace("fold", ""))
        for k in archive.files
    ))
    
    return [
        {
            "train": archive[f"fold{i}_train"],
            "val":   archive[f"fold{i}_val"],
            "test":  archive[f"fold{i}_test"],
        }
        for i in fold_indices
    ]

In [18]:
save_folds(mol_folds, "cmrt_folds.npz")

In [19]:
_mol_folds = load_folds("cmrt_folds.npz")
_mol_folds

[{'train': array([   0,    1,    2, ..., 3855, 3856, 3857], shape=(3478,)),
  'val': array([  56,   57,  140,  141,  160,  161,  214,  215,  222,  223,  250,
          251,  280,  281,  304,  305,  348,  349,  368,  369,  416,  417,
          428,  429,  446,  447,  464,  465,  470,  471,  478,  479,  496,
          497,  520,  521,  596,  597,  608,  609,  630,  631,  700,  701,
          786,  787,  838,  839,  944,  945,  966,  967, 1102, 1103, 1106,
         1107, 1260, 1261, 1280, 1281, 1326, 1327, 1366, 1367, 1372, 1373,
         1392, 1393, 1406, 1407, 1532, 1533, 1552, 1553, 1606, 1607, 1686,
         1687, 1780, 1781, 1892, 1893, 1948, 1949, 2010, 2011, 2022, 2023,
         2024, 2025, 2048, 2049, 2052, 2053, 2146, 2147, 2158, 2159, 2206,
         2207, 2304, 2305, 2312, 2313, 2320, 2321, 2330, 2331, 2450, 2451,
         2498, 2499, 2594, 2595, 2622, 2623, 2672, 2673, 2688, 2689, 2708,
         2709, 2718, 2719, 2728, 2729, 2734, 2735, 2736, 2737, 2768, 2769,
         2772, 27

In [20]:
df.iloc[_mol_folds[0]['val'], :]

,SMILES,SMILES_opp,TR/TE,F/L_class,@/@@_class,R/S_class
56,CC(C)(C)NC(=O)[C@H](OC(=O)C(C)(C)C)c1cccc([N+]...,CC(C)(C)NC(=O)[C@@H](OC(=O)C(C)(C)C)c1cccc([N+...,TE,F,@,R
57,CC(C)(C)NC(=O)[C@@H](OC(=O)C(C)(C)C)c1cccc([N+...,CC(C)(C)NC(=O)[C@H](OC(=O)C(C)(C)C)c1cccc([N+]...,TE,L,@@,S
140,Cc1cccc([C@H](NC(=O)OC(C)(C)C)C(=[N+]=[N-])P(=...,Cc1cccc([C@@H](NC(=O)OC(C)(C)C)C(=[N+]=[N-])P(...,TE,F,@,S
141,Cc1cccc([C@@H](NC(=O)OC(C)(C)C)C(=[N+]=[N-])P(...,Cc1cccc([C@H](NC(=O)OC(C)(C)C)C(=[N+]=[N-])P(=...,TE,L,@@,R
160,CCC[C@H](O)CCCC(=O)OCC,CCC[C@@H](O)CCCC(=O)OCC,TE,F,@,S
...,...,...,...,...,...,...
3673,O=C1Nc2ccccc2[C@]1(O)C#CCc1ccccc1,O=C1Nc2ccccc2[C@@]1(O)C#CCc1ccccc1,TR,L,@,R
3714,O=C[C@@](O)(CS(=O)(=O)c1nc2ccccc2s1)c1ccccc1,O=C[C@](O)(CS(=O)(=O)c1nc2ccccc2s1)c1ccccc1,TR,F,@@,R
3715,O=C[C@](O)(CS(=O)(=O)c1nc2ccccc2s1)c1ccccc1,O=C[C@@](O)(CS(=O)(=O)c1nc2ccccc2s1)c1ccccc1,TR,L,@,S
3802,O=S1(=O)N[C@](CCCc2ccccc2)(c2ccccc2)c2ccccc21,O=S1(=O)N[C@@](CCCc2ccccc2)(c2ccccc2)c2ccccc21,TR,L,@,R
